# Path Refinement Tutorial
This notebook demonstrates the PathGennie Path Refinement module using the Muller-Brown 2D toy potential.


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add the project root to the path so we can import pathrefinement
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from pathrefinement import MullerBrownPotential, PathRefiner, PathRefinementConfig
from pathrefinement.plotting import plot_initial_vs_refined, plot_refinement_iterations


## 1. Setup Potential and Initial Path
We initialize the Muller-Brown potential and generate a deliberately poor initial guess path from minimum A to minimum C.


In [ ]:
potential = MullerBrownPotential(energy_scale=0.1)
initial_path = potential.make_bad_initial_path("A", "C", n_images=20, noise=0.0)

# Let's see the potential shape and the initial path
fig, ax = plt.subplots(figsize=(6, 5))
from pathrefinement.plotting import plot_potential_surface, plot_path
cf = plot_potential_surface(potential, ax, xlim=(-2, 2), ylim=(-1, 2.5))
plot_path(initial_path, ax, color="red", label="Initial Path")
ax.legend()
plt.colorbar(cf, label="Energy")
plt.show()


## 2. Refine the Path
We configure the refiner to use PathGennie short-burst MD trajectories and neural network principal curve smoothing.


In [ ]:
config = PathRefinementConfig(
    n_iterations=5,
    n_trajectories=10,
    pathgennie_tau1=100,
    pathgennie_tau2=100,
    pathgennie_max_trial=10,
    pathgennie_max_cycle=100,
    nn_epochs=1000,
    keep_endpoints=True,
    seed=42,
    verbosity=1,
)

refiner = PathRefiner(potential, config)
result = refiner.refine(initial_path)


## 3. Visualize the Refined Path
We can plot the initial path vs the refined path, as well as the evolution of the path across iterations.


In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as tmpdir:
    comp_file = os.path.join(tmpdir, "comp.png")
    plot_initial_vs_refined(result, potential, comp_file)
    from IPython.display import Image, display
    display(Image(filename=comp_file))


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    iter_file = os.path.join(tmpdir, "iter.png")
    plot_refinement_iterations(result, potential, iter_file)
    display(Image(filename=iter_file))
